# Práctica 03 — Preprocesamiento lingüístico

**Curso:** NLP Clásico con Python · **Clase 03** · **Tiempo estimado:** 60-75 minutos
**Notebook de clase:** `notebooks_clase/clase03_preprocesamiento_linguistico.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Los textos de los ejemplos están en la celda de datos que sigue al setup; son cortos y están escritos para el curso (*ejemplo ilustrativo*).
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. La primera celda instala solo lo que falte (`nltk`, `spacy` y los modelos de spaCy en español e italiano).

---
## Instalación

Instala solo lo que falte y carga los modelos de spaCy.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("spacy", "spacy")
asegurar("nltk", "nltk")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")

import nltk
import spacy

try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords", quiet=True)

def cargar_modelo(nombre):
    """Carga un modelo de spaCy y lo descarga si no está instalado."""
    try:
        return spacy.load(nombre)
    except OSError:
        subprocess.run([sys.executable, "-m", "spacy", "download", nombre, "-q"], check=True)
        return spacy.load(nombre)

nlp_es = cargar_modelo("es_core_news_sm")
nlp_it = cargar_modelo("it_core_news_sm")
print("Recursos de NLTK y modelos de spaCy listos.")

---
## Celda 0 — Setup y datos

In [ ]:
import re
from collections import Counter

import pandas as pd
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

def tokenizar(texto):
    """Tokenización de la clase 2."""
    return re.findall(r"\w+", texto)

NOMBRE_NLTK = {"spa": "spanish", "ita": "italian"}
SW = {idioma: set(stopwords.words(nombre)) for idioma, nombre in NOMBRE_NLTK.items()}
STEMMER = {idioma: SnowballStemmer(nombre) for idioma, nombre in NOMBRE_NLTK.items()}
NLP = {"spa": nlp_es, "ita": nlp_it}
print("Setup listo.")

In [ ]:
# datos-es
NEGACIONES = {
    "spa": {"no", "nunca", "jamás", "ni", "sin", "nada", "nadie", "tampoco"},
    "ita": {"non", "mai", "né", "senza", "niente", "nessuno", "nulla", "neanche"},
}
DOMINIO = {"spa": {"pedido", "pedidos"}, "ita": {"ordine", "ordini"}}

TEXTO_EJ2 = {
    "spa": "Los pedidos llegaron tarde. El pedido llegó roto y pedimos otro pedido. Queremos devolver los pedidos rotos.",
    "ita": "Gli ordini sono arrivati tardi. L'ordine è arrivato rotto e abbiamo ordinato un altro ordine. Vogliamo restituire gli ordini rotti.",
}

PARES = {
    "spa": [("universidad", "universo", False), ("generar", "generación", True), ("generoso", "generar", False),
            ("venir", "viene", True), ("ir", "voy", True), ("pedido", "pedidos", True), ("cantar", "cantaron", True),
            ("organizar", "órgano", False), ("entrega", "entregar", True)],
    "ita": [("università", "universo", False), ("generare", "generazione", True), ("generoso", "generare", False),
            ("venire", "viene", True), ("andare", "vado", True), ("ordine", "ordini", True), ("cantare", "cantarono", True),
            ("consegna", "consegnare", True), ("ordine", "ordinare", False)],
}

TEXTO_EJ4 = {
    "spa": "Los clientes cobraron la factura y los paquetes llegaron tarde. Mi hermana vive en Madrid y comieron juntos.",
    "ita": "I clienti hanno pagato la fattura e i pacchi sono arrivati in ritardo. Mia sorella vive a Milano e hanno ordinato la cena.",
}

FRASES_RETO = {
    "spa": ["el pedido no llegó", "no está bien", "los pedidos llegaron rotos y no funcionan", "sin problemas con el envío",
            "nunca fue rápido el envío", "Ayer vino mi hermana y no compró nada"],
    "ita": ["l'ordine non è arrivato", "non è buono", "gli ordini sono arrivati rotti e non funzionano", "senza problemi con la spedizione",
            "la spedizione non è mai stata veloce", "Ieri è venuta mia sorella e non ha comprato niente"],
}

## Ejercicio 1 — Un filtro de stopwords que conserva las negaciones

Completa `filtrar_stopwords(tokens, idioma, conservar)`: devuelve los tokens que **no** están en la lista de NLTK (`SW[idioma]`), comparando en minúsculas, pero **conservando siempre** los que estén en el conjunto `conservar` (aunque estén en la lista). Devuelve los tokens originales, sin cambiar mayúsculas y respetando el orden. Con `conservar=()` se comporta como el filtro estándar.

Antes de programar, calcula a mano qué debería quedar de `no está bien` con el filtro estándar (recuerda: `no` y `está` son stopwords de NLTK) y con las negaciones conservadas.

In [ ]:
def filtrar_stopwords(tokens, idioma, conservar=()):
    resultado = None      # TODO: lista de tokens: fuera de SW[idioma] o dentro de conservar (comparando en minúsculas)
    return resultado

# Comprobación automática
assert filtrar_stopwords(["no", "está", "bien"], "spa") is not None, "Completa las líneas marcadas con TODO"  # dato-es
assert filtrar_stopwords(["no", "está", "bien"], "spa") == ["bien"], "sin conservar nada, el filtro estándar deja solo «bien»"  # dato-es
assert filtrar_stopwords(["no", "está", "bien"], "spa", NEGACIONES["spa"]) == ["no", "bien"], "con las negaciones conservadas quedan «no» y «bien»"  # dato-es
assert filtrar_stopwords(["non", "è", "buono"], "ita", NEGACIONES["ita"]) == ["non", "buono"], "en italiano: «non» y «buono»"
assert filtrar_stopwords(["Non", "è", "buono"], "ita", NEGACIONES["ita"]) == ["Non", "buono"], "compara en minúsculas pero devuelve el token original"
assert filtrar_stopwords(["sin", "problemas"], "spa") == ["problemas"] and filtrar_stopwords(["senza", "problemi"], "ita") == ["senza", "problemi"], "«sin» es stopword de NLTK; «senza» no"  # dato-es
assert filtrar_stopwords(tokenizar("nunca fue rápido"), "spa") == ["nunca", "rápido"], "«nunca» no está en la lista de NLTK"  # dato-es
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Cuánto se reduce el vocabulario con el stemming

En `TEXTO_EJ2` tienes un texto corto por idioma. Para cada uno, tokeniza con `tokenizar`, pasa a minúsculas y calcula:

- `tipos_spa`, `tipos_ita`: número de **tipos** (palabras distintas) del texto en minúsculas,
- `raices_spa`, `raices_ita`: número de **raíces distintas** tras aplicar el stemmer Snowball de cada idioma,
- `reduccion_spa`, `reduccion_ita`: reducción porcentual del vocabulario, `100 * (1 - raices / tipos)`, redondeada a 1 decimal.

Cuenta a mano los tipos del texto en español antes de programar (hay 17 tokens; `pedido` y `pedidos` son tipos distintos).

In [ ]:
tipos_spa = tipos_ita = None          # TODO: nº de palabras distintas en minúsculas
raices_spa = raices_ita = None        # TODO: nº de raíces distintas con Snowball
reduccion_spa = reduccion_ita = None  # TODO: 100 * (1 - raices / tipos), 1 decimal

# Comprobación automática
assert all(v is not None for v in (tipos_spa, tipos_ita, raices_spa, raices_ita, reduccion_spa, reduccion_ita)), "Completa las líneas marcadas con TODO"
assert (tipos_spa, tipos_ita) == (14, 18), "revisa los tipos: minúsculas y sin quitar stopwords"
assert (raices_spa, raices_ita) == (10, 14), "usa STEMMER[idioma].stem(palabra) sobre cada tipo"
assert (reduccion_spa, reduccion_ita) == (28.6, 22.2), "100 * (1 - raices / tipos) redondeado a 1 decimal"
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — Sobre-stemming y sub-stemming

`PARES[idioma]` contiene tripletas `(palabra_1, palabra_2, misma_palabra)`, donde `misma_palabra` indica si las dos formas son de la **misma** palabra (`True`) o de palabras distintas (`False`). Completa `clasificar_par` para que devuelva:

- `"sobre"` si las palabras son **distintas** pero tienen la **misma raíz** (sobre-stemming),
- `"sub"` si son de la **misma** palabra pero tienen **raíces distintas** (sub-stemming),
- `"ok"` en los otros dos casos.

Después rellena `etiquetas` con la etiqueta de cada par, en el orden de `PARES`.

In [ ]:
def clasificar_par(palabra_1, palabra_2, misma_palabra, idioma):
    raiz_1 = STEMMER[idioma].stem(palabra_1)
    raiz_2 = STEMMER[idioma].stem(palabra_2)
    etiqueta = None       # TODO: "sobre", "sub" o "ok" según las raíces y misma_palabra
    return etiqueta

etiquetas = {idioma: None for idioma in PARES}     # TODO: lista de etiquetas de cada idioma, en el orden de PARES

# Comprobación automática
assert all(v is not None for v in etiquetas.values()), "Completa las líneas marcadas con TODO"
assert etiquetas["spa"] == ["sobre", "ok", "sobre", "sub", "sub", "ok", "ok", "ok", "ok"], f"español: revisa {etiquetas['spa']}"
assert etiquetas["ita"] == ["sobre", "ok", "sobre", "sub", "sub", "ok", "ok", "ok", "sobre"], f"italiano: revisa {etiquetas['ita']}"
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Lemas de verbos con spaCy

Completa `lemas_verbos(texto, idioma)`: procesa el texto con `NLP[idioma]` y devuelve un diccionario `{forma_en_minúscula: lema}` solo con los tokens cuyo `pos_` sea `"VERB"`. Después comprueba en `TEXTO_EJ4` los verbos regulares comunes.

> **Nota:** la comprobación solo exige el infinitivo de unos pocos verbos regulares, que cualquier versión razonable de spaCy acierta. No comprobamos nada más porque los resultados de un modelo pequeño pueden cambiar con la versión.

In [ ]:
def lemas_verbos(texto, idioma):
    resultado = None      # TODO: diccionario {forma en minúscula: lema} de los tokens con pos_ == "VERB"
    return resultado

verbos_spa = lemas_verbos(TEXTO_EJ4["spa"], "spa")
verbos_ita = lemas_verbos(TEXTO_EJ4["ita"], "ita")

# Comprobación automática
assert verbos_spa is not None and verbos_ita is not None, "Completa las líneas marcadas con TODO"
for forma, lema in {"cobraron": "cobrar", "llegaron": "llegar", "comieron": "comer", "vive": "vivir"}.items():
    assert verbos_spa.get(forma) == lema, f"el lema de «{forma}» debería ser «{lema}»"
for forma, lema in {"pagato": "pagare", "arrivati": "arrivare", "vive": "vivere", "ordinato": "ordinare"}.items():
    assert verbos_ita.get(forma) == lema, f"el lema de «{forma}» debería ser «{lema}»"
assert "cliente" not in verbos_spa and "cliente" not in verbos_ita, "solo verbos: los sustantivos quedan fuera"
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Elegir el preprocesamiento para tres escenarios

Para cada escenario de `ESCENARIOS`, rellena en `PLAN` tres campos:

- `"stopwords"`: `"ninguna"` (no se quita nada), `"conservar_negaciones"` (lista de NLTK sin las negaciones) o `"estandar"` (lista de NLTK completa),
- `"normalizacion"`: `"ninguna"`, `"stemming"` o `"lematizacion"`,
- `"motivo"`: una frase (de al menos 20 caracteres) que justifique la elección.

No hay una única respuesta correcta: la comprobación solo exige los **vínculos** que impone cada tarea (véase el capítulo, sección «Guía de decisión»): un análisis de sentimiento no debe usar la lista estándar (se perderían las negaciones), y el reconocimiento de entidades y la generación de texto necesitan el texto sin tocar.

In [ ]:
ESCENARIOS = {
    "sentimiento": "Clasificar reseñas como positivas o negativas; muchas llevan negaciones («no está bien»).",
    "ner": "Encontrar nombres de personas y de empresas en correos.",
    "generacion": "Entrenar un modelo que genere respuestas de soporte en frases completas.",
}
PLAN = {
    escenario: {"stopwords": None, "normalizacion": None, "motivo": None}     # TODO: rellena los tres campos de cada escenario
    for escenario in ESCENARIOS
}

# Comprobación automática
OPC_SW = {"ninguna", "conservar_negaciones", "estandar"}
OPC_NORM = {"ninguna", "stemming", "lematizacion"}
for escenario, decision in PLAN.items():
    assert decision["stopwords"] in OPC_SW and decision["normalizacion"] in OPC_NORM, f"{escenario}: rellena stopwords y normalizacion con valores válidos"
    assert isinstance(decision["motivo"], str) and len(decision["motivo"]) >= 20, f"{escenario}: justifica la elección (al menos 20 caracteres)"
assert PLAN["sentimiento"]["stopwords"] != "estandar", "la lista estándar elimina «no» y «non»: cambia el sentimiento"
assert PLAN["ner"]["stopwords"] == "ninguna" and PLAN["ner"]["normalizacion"] == "ninguna", "el reconocimiento de entidades necesita el texto sin tocar"
assert PLAN["generacion"]["stopwords"] == "ninguna" and PLAN["generacion"]["normalizacion"] == "ninguna", "para generar texto no se quita ni se recorta nada"
print("Ejercicio 5 correcto ✔")

---
## Reto final — Una función `normalizar` con varios modos

Escribe `normalizar(texto, idioma, modo)` que devuelva una lista de tokens según el modo:

- `"minimo"`: tokeniza y pasa a minúsculas (ya está hecho),
- `"sin_stop"`: además, quita las stopwords con una **lista propia**: la de NLTK **menos** las negaciones de `NEGACIONES[idioma]` **más** las palabras de `DOMINIO[idioma]`,
- `"raices"`: como `"sin_stop"` y después stemming Snowball de cada token,
- `"lemas"`: lemas de spaCy (en minúscula) de los tokens alfanuméricos que no estén en la lista propia (el lematizador recibe el texto entero, no los tokens sueltos).

El test comprueba dos cosas sobre `FRASES_RETO`: que **ningún modo elimina nunca la negación** («no» en español, «non» en italiano) y que el vocabulario de `"raices"` es menor que el de `"minimo"`.

> **Nota honesta:** estas comprobaciones se hacen sobre seis frases por idioma: sirven para ver que la función se comporta, **no para saber qué modo es mejor**. Esa decisión se toma con datos reales y con validación (a partir de la clase 5).

In [ ]:
SW_PERS = {idioma: (SW[idioma] - NEGACIONES[idioma]) | DOMINIO[idioma] for idioma in SW}

def normalizar(texto, idioma, modo="minimo"):
    if modo == "minimo":
        return [t.lower() for t in tokenizar(texto)]
    # TODO 1: modo "sin_stop": minúsculas y fuera las stopwords de SW_PERS[idioma]
    # TODO 2: modo "raices": lo anterior más STEMMER[idioma].stem
    # TODO 3: modo "lemas": lemas de NLP[idioma](texto), en minúscula, de los tokens alfanuméricos fuera de SW_PERS
    return None

# Comprobación automática
NEG = {"spa": "no", "ita": "non"}
for idioma, frases in FRASES_RETO.items():
    for modo in ("minimo", "sin_stop", "raices", "lemas"):
        for frase in frases:
            resultado = normalizar(frase, idioma, modo)
            assert resultado is not None, f"el modo {modo} no está implementado"
            if NEG[idioma] in [t.lower() for t in tokenizar(frase)]:
                assert NEG[idioma] in resultado, f"el modo {modo} ha eliminado la negación en: {frase!r}"
    vocab_minimo = {t for f in frases for t in normalizar(f, idioma, "minimo")}
    vocab_raices = {t for f in frases for t in normalizar(f, idioma, "raices")}
    assert len(vocab_raices) < len(vocab_minimo), f"{idioma}: el vocabulario de raíces ({len(vocab_raices)}) debería ser menor que el mínimo ({len(vocab_minimo)})"
print("Reto final correcto ✔")

**Tus conclusiones (escribe 3-4 frases):** ¿qué modo de `normalizar` elegirías para clasificar tickets y cuál para analizar sentimiento? ¿Qué le pasaría a tu función con un texto ruidoso (erratas, alargamientos, clíticos)? ¿Cómo comprobarías, con datos separados de los que has usado para decidir, que tu elección es mejor?

*(Tu respuesta aquí.)*

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
def filtrar_stopwords(tokens, idioma, conservar=()):
    resultado = [t for t in tokens if t.lower() not in SW[idioma] or t.lower() in conservar]
    return resultado

assert filtrar_stopwords(["no", "está", "bien"], "spa") is not None, "Completa las líneas marcadas con TODO"  # dato-es
assert filtrar_stopwords(["no", "está", "bien"], "spa") == ["bien"], "sin conservar nada, el filtro estándar deja solo «bien»"  # dato-es
assert filtrar_stopwords(["no", "está", "bien"], "spa", NEGACIONES["spa"]) == ["no", "bien"], "con las negaciones conservadas quedan «no» y «bien»"  # dato-es
assert filtrar_stopwords(["non", "è", "buono"], "ita", NEGACIONES["ita"]) == ["non", "buono"], "en italiano: «non» y «buono»"
assert filtrar_stopwords(["Non", "è", "buono"], "ita", NEGACIONES["ita"]) == ["Non", "buono"], "compara en minúsculas pero devuelve el token original"
assert filtrar_stopwords(["sin", "problemas"], "spa") == ["problemas"] and filtrar_stopwords(["senza", "problemi"], "ita") == ["senza", "problemi"], "«sin» es stopword de NLTK; «senza» no"  # dato-es
assert filtrar_stopwords(tokenizar("nunca fue rápido"), "spa") == ["nunca", "rápido"], "«nunca» no está en la lista de NLTK"  # dato-es

### Solución del ejercicio 2

In [ ]:
def contar(texto, idioma):
    tipos = {t.lower() for t in tokenizar(texto)}
    raices = {STEMMER[idioma].stem(t) for t in tipos}
    return len(tipos), len(raices)

tipos_spa, raices_spa = contar(TEXTO_EJ2["spa"], "spa")
tipos_ita, raices_ita = contar(TEXTO_EJ2["ita"], "ita")
reduccion_spa = round(100 * (1 - raices_spa / tipos_spa), 1)
reduccion_ita = round(100 * (1 - raices_ita / tipos_ita), 1)

assert all(v is not None for v in (tipos_spa, tipos_ita, raices_spa, raices_ita, reduccion_spa, reduccion_ita)), "Completa las líneas marcadas con TODO"
assert (tipos_spa, tipos_ita) == (14, 18), "revisa los tipos: minúsculas y sin quitar stopwords"
assert (raices_spa, raices_ita) == (10, 14), "usa STEMMER[idioma].stem(palabra) sobre cada tipo"
assert (reduccion_spa, reduccion_ita) == (28.6, 22.2), "100 * (1 - raices / tipos) redondeado a 1 decimal"

### Solución del ejercicio 3

In [ ]:
def clasificar_par(palabra_1, palabra_2, misma_palabra, idioma):
    raiz_1 = STEMMER[idioma].stem(palabra_1)
    raiz_2 = STEMMER[idioma].stem(palabra_2)
    if raiz_1 == raiz_2 and not misma_palabra:
        etiqueta = "sobre"
    elif raiz_1 != raiz_2 and misma_palabra:
        etiqueta = "sub"
    else:
        etiqueta = "ok"
    return etiqueta

etiquetas = {idioma: [clasificar_par(a, b, misma, idioma) for a, b, misma in pares] for idioma, pares in PARES.items()}

assert all(v is not None for v in etiquetas.values()), "Completa las líneas marcadas con TODO"
assert etiquetas["spa"] == ["sobre", "ok", "sobre", "sub", "sub", "ok", "ok", "ok", "ok"], f"español: revisa {etiquetas['spa']}"
assert etiquetas["ita"] == ["sobre", "ok", "sobre", "sub", "sub", "ok", "ok", "ok", "sobre"], f"italiano: revisa {etiquetas['ita']}"

### Solución del ejercicio 4

In [ ]:
def lemas_verbos(texto, idioma):
    resultado = {t.text.lower(): t.lemma_ for t in NLP[idioma](texto) if t.pos_ == "VERB"}
    return resultado

verbos_spa = lemas_verbos(TEXTO_EJ4["spa"], "spa")
verbos_ita = lemas_verbos(TEXTO_EJ4["ita"], "ita")

assert verbos_spa is not None and verbos_ita is not None, "Completa las líneas marcadas con TODO"
for forma, lema in {"cobraron": "cobrar", "llegaron": "llegar", "comieron": "comer", "vive": "vivir"}.items():
    assert verbos_spa.get(forma) == lema, f"el lema de «{forma}» debería ser «{lema}»"
for forma, lema in {"pagato": "pagare", "arrivati": "arrivare", "vive": "vivere", "ordinato": "ordinare"}.items():
    assert verbos_ita.get(forma) == lema, f"el lema de «{forma}» debería ser «{lema}»"
assert "cliente" not in verbos_spa and "cliente" not in verbos_ita, "solo verbos: los sustantivos quedan fuera"

### Solución del ejercicio 5

In [ ]:
ESCENARIOS = {
    "sentimiento": "Clasificar reseñas como positivas o negativas; muchas llevan negaciones («no está bien»).",
    "ner": "Encontrar nombres de personas y de empresas en correos.",
    "generacion": "Entrenar un modelo que genere respuestas de soporte en frases completas.",
}
PLAN = {
    "sentimiento": {"stopwords": "conservar_negaciones", "normalizacion": "lematizacion",
                     "motivo": "Las negaciones invierten el sentido; el lema une formas de la misma palabra."},
    "ner": {"stopwords": "ninguna", "normalizacion": "ninguna",
            "motivo": "Las mayúsculas y el contexto delatan los nombres propios; no hay que tocar el texto."},
    "generacion": {"stopwords": "ninguna", "normalizacion": "ninguna",
                  "motivo": "El texto generado debe ser texto completo y correcto, con todas sus palabras."},
}

OPC_SW = {"ninguna", "conservar_negaciones", "estandar"}
OPC_NORM = {"ninguna", "stemming", "lematizacion"}
for escenario, decision in PLAN.items():
    assert decision["stopwords"] in OPC_SW and decision["normalizacion"] in OPC_NORM, f"{escenario}: rellena stopwords y normalizacion con valores válidos"
    assert isinstance(decision["motivo"], str) and len(decision["motivo"]) >= 20, f"{escenario}: justifica la elección (al menos 20 caracteres)"
assert PLAN["sentimiento"]["stopwords"] != "estandar", "la lista estándar elimina «no» y «non»: cambia el sentimiento"
assert PLAN["ner"]["stopwords"] == "ninguna" and PLAN["ner"]["normalizacion"] == "ninguna", "el reconocimiento de entidades necesita el texto sin tocar"
assert PLAN["generacion"]["stopwords"] == "ninguna" and PLAN["generacion"]["normalizacion"] == "ninguna", "para generar texto no se quita ni se recorta nada"

### Solución del reto final

In [ ]:
SW_PERS = {idioma: (SW[idioma] - NEGACIONES[idioma]) | DOMINIO[idioma] for idioma in SW}

def normalizar(texto, idioma, modo="minimo"):
    if modo == "minimo":
        return [t.lower() for t in tokenizar(texto)]
    if modo == "lemas":
        return [t.lemma_.lower() for t in NLP[idioma](texto)
                if re.fullmatch(r"\w+", t.text) and t.text.lower() not in SW_PERS[idioma]]
    tokens = [t.lower() for t in tokenizar(texto) if t.lower() not in SW_PERS[idioma]]
    if modo == "raices":
        tokens = [STEMMER[idioma].stem(t) for t in tokens]
    return tokens

NEG = {"spa": "no", "ita": "non"}
for idioma, frases in FRASES_RETO.items():
    for modo in ("minimo", "sin_stop", "raices", "lemas"):
        for frase in frases:
            resultado = normalizar(frase, idioma, modo)
            assert resultado is not None, f"el modo {modo} no está implementado"
            if NEG[idioma] in [t.lower() for t in tokenizar(frase)]:
                assert NEG[idioma] in resultado, f"el modo {modo} ha eliminado la negación en: {frase!r}"
    vocab_minimo = {t for f in frases for t in normalizar(f, idioma, "minimo")}
    vocab_raices = {t for f in frases for t in normalizar(f, idioma, "raices")}
    assert len(vocab_raices) < len(vocab_minimo), f"{idioma}: el vocabulario de raíces ({len(vocab_raices)}) debería ser menor que el mínimo ({len(vocab_minimo)})"
print("Reto final correcto ✔")

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto no te ha quedado claro (stopwords y negaciones, sobre-stemming, sub-stemming, POS y lema, qué preprocesar según la tarea)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*